# DSC360 Exercise 8.2

**Author:** James Sneddon

**Date:** May 9, 2026

**Modified By:** James Sneddon

In [1]:
import pandas as pd
import nltk
import spacy
import sklearn_crfsuite
from sklearn.model_selection import train_test_split
from sklearn_crfsuite import metrics
import joblib

## EDA

In [2]:
df = pd.read_csv('data/ner_database.csv', encoding='ISO-8859-1')
df['Sentence #'] = df['Sentence #'].ffill()
df = df.dropna(subset=['Word'])

In [3]:
n_sentences = df['Sentence #'].nunique()
n_words     = df['Word'].nunique()
n_pos       = df['POS'].nunique()
n_tags      = df['Tag'].nunique()

print(f'Total sentences : {n_sentences:,}')
print(f'Unique words    : {n_words:,}')
print(f'Unique POS tags : {n_pos:,}')
print(f'Unique NER tags : {n_tags:,}')

print()
print(df['Tag'].value_counts())

Total sentences : 47,959
Unique words    : 35,177
Unique POS tags : 42
Unique NER tags : 17

Tag
O        887898
B-geo     37644
B-tim     20333
B-org     20143
I-per     17251
B-per     16990
I-org     16784
B-gpe     15870
I-geo      7414
I-tim      6528
B-art       402
B-eve       308
I-art       297
I-eve       253
B-nat       201
I-gpe       198
I-nat        51
Name: count, dtype: int64


## Preprocessing and Feature Extraction

In [4]:
def word2features(sent, i):
    word = sent[i][0]
    postag = sent[i][1]

    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        'postag': postag,
        'postag[:2]': postag[:2],
    }

    if i > 0:
        word1 = sent[i - 1][0]
        postag1 = sent[i - 1][1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
            '-1:postag': postag1,
            '-1:postag[:2]': postag1[:2],
        })
    else:
        features['BOS'] = True

    if i < len(sent) - 1:
        word1 = sent[i + 1][0]
        postag1 = sent[i + 1][1]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
            '+1:postag': postag1,
            '+1:postag[:2]': postag1[:2],
        })
    else:
        features['EOS'] = True

    return features


def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]


def sent2labels(sent):
    return [label for _, _, label in sent]


sentences = df.groupby('Sentence #').apply(
    lambda s: list(zip(s['Word'].values, s['POS'].values, s['Tag'].values)),
    include_groups=False
).tolist()

X = [sent2features(s) for s in sentences]
y = [sent2labels(s) for s in sentences]

print(f'Total samples : {len(X):,}')

Total samples : 47,959


## Model

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f'Train size : {len(X_train):,}')
print(f'Test size  : {len(X_test):,}')

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
    verbose=True
)

crf.fit(X_train, y_train)
joblib.dump(crf, 'ner_model.pkl')

Train size : 35,969
Test size  : 11,990


loading training data to CRFsuite: 100%|█| 35969/35969 [00:04<00:00, 7644.73it/s



Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 133629
Seconds required: 1.579

L-BFGS optimization
c1: 0.100000
c2: 0.100000
num_memories: 6
max_iterations: 100
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

Iter 1   time=1.19  loss=1264023.14 active=132637 feature_norm=1.00
Iter 2   time=1.22  loss=994061.26 active=131294 feature_norm=4.42
Iter 3   time=0.62  loss=776413.97 active=125970 feature_norm=3.87
Iter 4   time=3.19  loss=422140.04 active=127018 feature_norm=3.24
Iter 5   time=0.64  loss=355771.56 active=129030 feature_norm=4.04
Iter 6   time=0.65  loss=264114.80 active=124042 feature_norm=6.10
Iter 7   time=0.62  loss=222292.39 active=117180 feature_norm=7.69
Iter 8   time=0.62  loss=197819.80 active=110837 feature_norm=8.75
Iter 9   time=0.65  loss=176880.09 active=105647 feature_norm

['ner_model.pkl']

## Model Performance

In [6]:
y_pred = crf.predict(X_test)

labels = list(crf.classes_)
labels.remove('O')

print(metrics.flat_classification_report(y_test, y_pred, labels=labels))

              precision    recall  f1-score   support

       B-org       0.81      0.73      0.77      5116
       B-per       0.85      0.83      0.84      4239
       I-per       0.85      0.90      0.88      4273
       B-geo       0.86      0.91      0.89      9403
       I-geo       0.81      0.80      0.81      1826
       B-tim       0.93      0.89      0.91      5095
       I-org       0.82      0.80      0.81      4195
       B-gpe       0.97      0.94      0.96      3961
       I-tim       0.84      0.82      0.83      1604
       B-nat       0.58      0.25      0.35        55
       B-eve       0.51      0.34      0.41        80
       B-art       0.35      0.14      0.20       102
       I-art       0.25      0.07      0.11        90
       I-eve       0.43      0.20      0.28        74
       I-gpe       0.86      0.53      0.66        36
       I-nat       0.57      0.22      0.32        18

   micro avg       0.86      0.85      0.86     40167
   macro avg       0.71   

## Exercise 1: Custom CRF Tagger

In [7]:
test_sentence = 'Fourteen days ago, Emperor Palpatine left San Diego, CA for Tatooine to follow Luke Skywalker.'

tokens   = nltk.word_tokenize(test_sentence)
pos_tags = nltk.pos_tag(tokens)

X_test_sentence = sent2features(pos_tags)
y_pred_sentence = crf.predict([X_test_sentence])[0]

results_df = pd.DataFrame({'Token': tokens, 'Tag': y_pred_sentence})
print(results_df.to_string(index=False))

print()
print('Named entities (CRF):')
for tok, tag in zip(tokens, y_pred_sentence):
    if tag != 'O':
        print(f'  {tok:<20} {tag}')

    Token   Tag
 Fourteen B-per
     days     O
      ago     O
        ,     O
  Emperor B-per
Palpatine I-per
     left     O
      San B-geo
    Diego I-geo
        ,     O
       CA B-org
      for I-org
 Tatooine I-org
       to     O
   follow     O
     Luke B-per
Skywalker I-per
        .     O

Named entities (CRF):
  Fourteen             B-per
  Emperor              B-per
  Palpatine            I-per
  San                  B-geo
  Diego                I-geo
  CA                   B-org
  for                  I-org
  Tatooine             I-org
  Luke                 B-per
  Skywalker            I-per


## Exercise 2: spaCy NER

In [8]:
nlp = spacy.load('en_core_web_sm')
doc = nlp(test_sentence)

spacy_df = pd.DataFrame(
    [(token.text, token.ent_type_) for token in doc if token.ent_type_],
    columns=['Token', 'Entity Type']
)
print(spacy_df.to_string(index=False))

print()
print('Named entities (spaCy):')
for ent in doc.ents:
    print(f'  {ent.text:<30} {ent.label_}')

    Token Entity Type
 Fourteen        DATE
     days        DATE
      ago        DATE
Palpatine      PERSON
      San         GPE
    Diego         GPE
       CA WORK_OF_ART
      for WORK_OF_ART
 Tatooine WORK_OF_ART
     Luke      PERSON
Skywalker      PERSON

Named entities (spaCy):
  Fourteen days ago              DATE
  Palpatine                      PERSON
  San Diego                      GPE
  CA for Tatooine                WORK_OF_ART
  Luke Skywalker                 PERSON


## Exercise 3: Compare and Contrast

The CRF uses IOB2 labels from the GMB corpus; spaCy uses flat OntoNotes labels. Both models struggle with Star Wars proper nouns absent from their training data, but spaCy generalizes better to multi-token spans and catches the time expression.